# S2: In-hand cube reorientation (LEAP hand, MJX)


PPO on in-hand cube reorientation with the LEAP hand, in MJX.

A 5 cm cube starts held in a partially closed hand. A target orientation is
sampled each episode; the policy commands 16 finger joints and must rotate the
cube to within 0.1 rad without dropping it. Roughly 10^8 environment steps.

**This is the only project in the portfolio that has never trained.** It was
written and unit-tested on a machine with no GPU, where one `mjx.step` costs
about 4 seconds. The smoke gate in section 2 is not a formality: until it
prints an iteration line, the training loop has never closed anywhere.

Designed to be run many times. Each session resumes from the newest checkpoint
in Drive.


---

### Before you run anything

1. **Runtime → Change runtime type → T4 GPU.** Every cell below assumes it.
2. **Keep this tab visible.** Free Colab disconnects an idle notebook after
   about 90 minutes and reclaims the runtime; `/content` does not survive it.
3. **The free tier has a quota you cannot see.** It is not published, it
   varies, and it is consumed by wall-clock GPU time whether or not you are
   computing. Expect a few hours a day, and expect to be cut off mid-run
   without warning. Every long-running cell here is written to survive that.

Checkpoints go to Google Drive, not to `/content`. That is the whole reason
the Drive cell exists, a run that checkpoints only to local disk loses
everything the moment the runtime is reclaimed, which on free Colab is the
normal way a session ends rather than an exceptional one.

## 1. Hardware, packages, Drive, code

In [ ]:
# --- what hardware did we actually get? ---
import subprocess, sys
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                      '--format=csv'], capture_output=True, text=True).stdout)

# A label is not hardware. A Kaggle session advertised as "T4 x2" reported a
# P100 to the driver, which has compute capability 6.0 and cannot run several
# things a T4 can. Record what the driver says and quote it as such; never
# write "measured on a T4" because the runtime menu said T4.

In [ ]:
# --- packages ---
# jax with CUDA is preinstalled on Colab GPU runtimes. mujoco-mjx is not, and
# installing it can drag in a CPU-only jax wheel that silently replaces the
# working one. So: install, then re-check the device, and only reinstall jax
# if the check fails.
import subprocess, sys

def sh(cmd):
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True)
    if r.returncode:
        raise SystemExit(f'command failed: {cmd}')

sh(f'{sys.executable} -m pip install -q mujoco mujoco-mjx optax')

import importlib, jax
importlib.reload(jax)
if not any(d.platform == 'gpu' for d in jax.devices()):
    print('jax lost the GPU during install; reinstalling the CUDA wheel')
    sh(f'{sys.executable} -m pip install -q -U "jax[cuda12]"')
    raise SystemExit(
        'Reinstalled jax. Runtime -> Restart session, then run this cell '
        'again. (A restart is required: the CPU-only jax is already imported '
        'into this process and reimporting will not replace it.)')

In [ ]:
import jax, mujoco
print('jax     ', jax.__version__)
print('mujoco  ', mujoco.__version__)
print('devices ', jax.devices())
print('kind    ', getattr(jax.devices()[0], 'device_kind', '?'), '(as reported by the driver)')

# Hard stop, not a warning. On CPU a single mjx.step of the LEAP scene costs
# about 4 seconds and the compile runs past half an hour: a CPU session is not
# a slow run, it is no run.
assert any(d.platform == 'gpu' for d in jax.devices()), \
    'No GPU. Runtime -> Change runtime type -> T4 GPU, then restart and re-run.'
print()
print('GPU OK')

In [ ]:
# --- Drive, for anything that must outlive this runtime ---
import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive/robotics-rl-portfolio')
DRIVE.mkdir(parents=True, exist_ok=True)
WORK = Path('/content/work'); WORK.mkdir(exist_ok=True)

# One XLA compilation cache on Drive, shared by this process and every
# subprocess it starts. MJX compiles the LEAP scene in minutes and a
# preempted run re-pays that every session; cached, it is seconds. Set as
# environment variables rather than jax.config so child processes inherit it,
# and because the directory has to be known before the first compile.
os.environ['JAX_COMPILATION_CACHE_DIR'] = str(DRIVE / 'jax_cache')
os.environ['JAX_PERSISTENT_CACHE_MIN_COMPILE_TIME_SECS'] = '0.5'
os.environ['JAX_COMPILATION_CACHE_MAX_SIZE'] = str(2_000_000_000)
(DRIVE / 'jax_cache').mkdir(parents=True, exist_ok=True)

_n = len([p for p in (DRIVE / 'jax_cache').rglob('*') if p.is_file()])
print('drive :', DRIVE)
print('local :', WORK)
print('cache :', _n, 'entries', '(empty: this session pays full compile cost)' if not _n else '')

In [ ]:
# --- code ---
import os, shutil, subprocess, sys
from pathlib import Path

SRC = WORK / 'robotics-rl-portfolio'
REPO = 'https://github.com/JacobEGarcia/robotics-rl-portfolio.git'

def sh(cmd, **kw):
    print('$', cmd, flush=True)
    return subprocess.run(cmd, shell=True, check=True, **kw)

if SRC.exists():
    sh(f'cd {SRC} && git pull --ff-only')
else:
    sh(f'git clone --depth 1 {REPO} {SRC}')

need = SRC / 's2_inhand/train.py'
if not need.exists():
    # Fallback for code that is committed locally but not pushed. Tar the
    # repo on your machine, drop it in Drive, and this picks it up:
    #   tar czf portfolio.tgz --exclude=assets --exclude=runs .
    tgz = DRIVE / 'portfolio.tgz'
    if tgz.exists():
        print(f'{need} missing from the clone; unpacking {tgz} over it')
        sh(f'tar xzf {tgz} -C {SRC}')
    if not need.exists():
        raise SystemExit(
            f'{need} is not in the cloned repo and no {tgz} was found.\n'
            f'Push it from your machine:\n'
            f'    cd ~/Downloads/hermestes/robotics-rl-portfolio && git push origin main\n'
            f'or upload a tarball to {tgz}.')

os.environ['PYTHONPATH'] = str(SRC)
sys.path.insert(0, str(SRC))
print()
print('code OK at', SRC)

In [ ]:
# --- the robot models ---
# assets/menagerie is gitignored in the portfolio repo on purpose: Menagerie
# is 2.3 GB of third-party assets and is itself a git repo, so it is fetched
# rather than vendored. A sparse checkout gets what is needed in a few
# seconds instead of pulling all of it.
MENAGERIE = SRC / 'assets' / 'menagerie'
WANT = ['leap_hand', 'franka_emika_panda', 'unitree_z1',
        'unitree_go1', 'unitree_h1', 'shadow_hand', 'dynamixel_2r']

if not (MENAGERIE / 'leap_hand' / 'right_hand.xml').exists():
    shutil.rmtree(MENAGERIE, ignore_errors=True)
    MENAGERIE.parent.mkdir(parents=True, exist_ok=True)
    sh('git clone --depth 1 --filter=blob:none --sparse '
       'https://github.com/google-deepmind/mujoco_menagerie.git ' + str(MENAGERIE))
    sh(f'cd {MENAGERIE} && git sparse-checkout set ' + ' '.join(WANT))

missing = [w for w in WANT if not (MENAGERIE / w).exists()]
assert not missing, f'sparse checkout did not produce: {missing}'
print('models OK:', sorted(p.name for p in MENAGERIE.iterdir() if p.is_dir())[:12])

## 2. Does the scene still hold the cube?

Loads `scene.xml` and re-checks the reset grasp. A broken or partial asset fetch shows up here in seconds instead of twenty minutes into a compile.

Initial penetration is a cliff, not a gradient: a configuration starting the cube 10 mm or more inside a finger geom does not settle badly, it ejects the cube at tens of metres per second on the first step. Both numbers are checked.

In [ ]:
import mujoco, numpy as np
m = mujoco.MjModel.from_xml_path(str(SRC / 's2_inhand' / 'scene.xml'))
d = mujoco.MjData(m)
mujoco.mj_resetDataKeyframe(m, d, 0)
mujoco.mj_forward(m, d)

cb = mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_BODY, 'cube')
cg = mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_GEOM, 'cube_geom')
pen = min((d.contact[c].dist for c in range(d.ncon)
           if cg in (d.contact[c].geom1, d.contact[c].geom2)), default=0.0)
start = d.xpos[cb].copy()
for _ in range(2500):
    mujoco.mj_step(m, d)
drift = float(np.linalg.norm(d.xpos[cb] - start))

print(f'nq={m.nq} nv={m.nv} nu={m.nu} ngeom={m.ngeom}')
print(f'reset penetration {pen*1e3:.3f} mm   drift over 5 s {drift*1e3:.2f} mm')
assert abs(pen) < 0.010, 'penetration at reset is in the ejection regime'
assert drift < 0.03, 'grasp is not holding; do not train against this'
print('scene OK')

### Smoke gate, do not skip

Four environments, two iterations, one checkpoint written and reloaded. It proves the rollout, the GAE, the PPO update and the checkpoint path close end to end **on this machine**.

It must print at least one `step ...` line and one `checkpoint @ ...` line. Two minutes here against an afternoon of a run that was never going to work.

In [ ]:
import os, subprocess, sys, time
os.chdir(SRC)
SMOKE = WORK / 'smoke_ckpt'

t0 = time.time()
r = subprocess.run([sys.executable, '-u', '-m', 's2_inhand.train', '--smoke',
                    '--ckpt-dir', str(SMOKE)],
                   env=dict(os.environ, PYTHONPATH=str(SRC)),
                   capture_output=True, text=True)
print(r.stdout[-4000:])
if r.stderr.strip():
    print('--- stderr ---'); print(r.stderr[-3000:])
print(f'\nexit={r.returncode} in {time.time()-t0:.0f}s')

wrote = list(SMOKE.glob('ckpt_*.pkl'))
assert r.returncode == 0, 'smoke run failed, see stderr'
assert wrote, 'smoke run wrote no checkpoint'
assert 'step ' in r.stdout, 'smoke run never closed an iteration'
print('SMOKE GATE PASSED ->', [p.name for p in wrote])

## 3. Size the batch by measurement, not by guess

Reuses S10's throughput harness on this exact scene. Picks the **knee**, the smallest batch within 10% of the best throughput, not the largest batch that fits. Past the knee each extra environment buys under 10% while costing proportional memory and a longer compile, and on a preemptible session a longer compile is a direct loss.

Takes 5-15 minutes and saves considerably more than that.

In [ ]:
from s10_gpu_scaling import throughput

pts = throughput.mjx_sweep(m, [256, 512, 1024, 2048, 4096, 8192], n_steps=50)
k = throughput.knee(pts)
best = max((p for p in pts if p.steps_per_s), key=lambda p: p.steps_per_s, default=None)

NUM_ENVS = k.n_envs if k else 1024
print()
print(f'chosen --num-envs {NUM_ENVS}')
if best:
    print(f'best {best.steps_per_s:,.0f} steps/s at n={best.n_envs:,}')
    print(f'1e8 steps at the knee ~ {1e8/k.steps_per_s/3600:.1f} GPU-hours')
    print(f'  = {1e8/k.steps_per_s/3600/2.5:.1f} sessions at 2.5 h each')

## 4. Train

`--resume` searches Drive as well as local disk, so this is the same cell every session. `--max-hours 2.5` makes the trainer write a final checkpoint and exit cleanly before the free tier is likely to cut it off; being killed between checkpoints throws away everything since the last one.

**Watch the `curr` column, not the reward.** If the curriculum is not widening after a few million steps, the reward is not shaping behaviour and the fix is the reward, not more steps. That is the expected failure mode for this project and it is worth catching in hour one rather than hour eleven.

In [ ]:
CKPT_LOCAL  = WORK / 'checkpoints'
CKPT_DRIVE  = DRIVE / 's2_checkpoints'
TOTAL_STEPS = 100_000_000
MAX_HOURS   = 2.5

cmd = [sys.executable, '-u', '-m', 's2_inhand.train',
       '--total-steps', str(TOTAL_STEPS),
       '--num-envs', str(NUM_ENVS),
       '--ckpt-dir', str(CKPT_LOCAL),
       '--ckpt-mirror', str(CKPT_DRIVE),
       '--ckpt-every', '2000000',
       '--max-hours', str(MAX_HOURS),
       '--resume']
print('$', ' '.join(cmd), flush=True)

# Stream the output. A multi-hour run that prints nothing until it exits
# cannot be babysat, and the point is to notice a flat curriculum early.
p = subprocess.Popen(cmd, env=dict(os.environ, PYTHONPATH=str(SRC)),
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                     text=True, bufsize=1)
try:
    for line in p.stdout:
        print(line, end='', flush=True)
except KeyboardInterrupt:
    p.terminate()
    print('\ninterrupted; the newest checkpoint is already in Drive')
p.wait()

## 5. Where the run is

In [ ]:
import json
for name, d in (('local', CKPT_LOCAL), ('drive', CKPT_DRIVE)):
    cks = sorted(d.glob('ckpt_*.pkl')) if d.exists() else []
    print(f'{name:6} {len(cks)} checkpoint(s)',
          [f'{c.name} ({c.stat().st_size/1e6:.1f} MB)' for c in cks[-3:]])
meta = CKPT_DRIVE / 'latest.json'
if meta.exists():
    print(); print(meta.read_text())
print()
print('To continue: new session, run sections 1 and 4. Skip 2 and 3.')